# 第 5 章 · 计算机视觉：卷积从手算到训练

> 对应正文：[CNN 卷积神经网络](../docs/4-专业方向/05-计算机视觉/01-CNN卷积神经网络.md) · [返回本章 README](../docs/4-专业方向/05-计算机视觉/README.md) · [返回 notebooks 总览](./README.md)

本 notebook 把正文"动手试试"与两个"数学深潜"扩展成 7 个实验：卷积手算复现、多通道卷积形状流、卷积核动物园、最大池化、感受野计算器、小 CNN 训手写数字（torch 版 + sklearn 离线替身）、残差连接的梯度高速公路。

**环境说明**

- 实验 1~5、7 只依赖 numpy + matplotlib，实验 6B 用 sklearn 内置 digits（无外部下载），全部离线可跑、随机种子固定；
- 实验 6A 的 PyTorch/CNN 部分需本地环境（`pip install torch torchvision`），代码已按标准写法给出并注明本地运行预期（MNIST 测试集 98%+）；未安装 torch 时自动跳过，不影响其余单元。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 固定随机种子，保证每次运行结果一致
np.random.seed(42)

# matplotlib 中文显示设置（Windows 用 SimHei/微软雅黑；Mac 可换 Arial Unicode MS）
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False    # 让负号正常显示
# 对数轴刻度走 mathtext：把数学字体钉回 DejaVu Sans，避免中文字体缺负号字形（U+2212）
plt.rcParams["mathtext.fontset"] = "dejavusans"
for _k in ("mathtext.rm", "mathtext.it", "mathtext.bf", "mathtext.sf", "mathtext.tt", "mathtext.cal"):
    plt.rcParams[_k] = "DejaVu Sans"

print("numpy 版本:", np.__version__, "；matplotlib 中文字体已设置")

## 实验 1：卷积手算复现——5×5 图 × 3×3 竖边缘核

**目标**：正文算例的可跑版：左边两列暗、右边三列亮的 5×5 图，过"左暗右亮"竖边缘核，输出应为 3×3 的 [3 3 0; 3 3 0; 3 3 0]。再滑一枚水平边缘核对比——方向选择性：竖条纹对水平核"失明"。

In [ ]:
# ========= 实验 1：卷积手算复现 =========
def conv2d_valid(image, kernel):
    """单通道卷积（工程实现 = 互相关：不翻转核、直接对位乘加），不补零、步幅 1"""
    H, W = image.shape
    k = kernel.shape[0]
    out = np.zeros((H - k + 1, W - k + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            window = image[i:i + k, j:j + k]          # 取出窗口
            out[i, j] = np.sum(window * kernel)       # 对位相乘再求和 = 正文手算的那一步
    return out

I = np.array([[0, 0, 1, 1, 1]] * 5, dtype=float)      # 左两列暗、右三列亮：一条竖直边缘
K_v = np.array([[-1, 0, 1],
                [-1, 0, 1],
                [-1, 0, 1]], dtype=float)             # 竖边缘核（左暗右亮 → 正分）
K_h = np.array([[-1, -1, -1],
                [ 0,  0,  0],
                [ 1,  1,  1]], dtype=float)           # 水平边缘核（上暗下亮 → 正分）

out_v = conv2d_valid(I, K_v)
print("输入图 I（5×5）：")
print(I.astype(int))
print("\n竖边缘核输出（3×3）：")
print(out_v.astype(int))
print("← 与正文手算完全一致：第 1 个窗口每行 0×(-1)+0×0+1×1 = 1，三行合计 3；")
print("  窗口里有明暗交界就得 3，全亮得 0 —— 输出亮的格子正好标出边缘在哪")

out_h = conv2d_valid(I, K_h)
print("\n水平边缘核输出（3×3）：")
print(out_h.astype(int))
print("← 全 0：输入只有竖直边缘，水平核对它完全『失明』——这就是方向选择性，")
print("  所以每层要同时配多方向的核")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
axes[0].imshow(I, cmap="gray");    axes[0].set_title("输入图 I（5×5，左暗右亮）")
axes[1].imshow(out_v, cmap="gray"); axes[1].set_title("竖边缘核响应（3 3 0 / 3 3 0 / 3 3 0）")
axes[2].imshow(out_h, cmap="gray"); axes[2].set_title("水平边缘核响应（全 0：失明）")
for ax in axes:
    ax.set_xlabel("列 →"); ax.set_ylabel("行 ↓")
plt.tight_layout(); plt.show()

**小结**：卷积输出与正文手算逐位一致（[3 3 0] × 3）；水平核输出全 0 印证了方向选择性。注意工程实现的"卷积"其实是互相关（核不翻转、直接对位乘加）——核是学出来的，翻不翻转只是参数排列不同，`conv2d_valid` 采用与 PyTorch 相同的约定。

## 实验 2：多通道卷积形状流——C_in=3 → C_out=2

**目标**：正文③的可跑版：卷积核不是 3×3 的薄片，而是 3×3×C_in 的"小方块"。逐步打印"输入 [3,5,5] → 核 [2,3,3,3] → 每个输入通道的中间打分图 → 逐位置相加加偏置 → 输出 [2,3,3]"，并用参数量公式 (K·K·C_in + 1) × C_out 核对。

In [ ]:
# ========= 实验 2：多通道卷积形状流 =========
rng = np.random.default_rng(0)
img3 = rng.uniform(0, 1, (3, 5, 5))                    # 假彩色图：3 通道 × 5×5
C_in, C_out, k = 3, 2, 3
kernels3 = rng.normal(0, 0.5, (C_out, C_in, k, k))     # 2 个『3×3×3 小方块』
biases3 = np.array([0.1, -0.2])                        # 每个输出通道一个偏置

print("输入形状   img3    =", img3.shape, "  ← [C_in=3, 5, 5]")
print("卷积核形状 kernels =", kernels3.shape, " ← [C_out=2, C_in=3, 3, 3]")
print("每个输出通道的打分图 = 该核在 3 个输入通道各滑一遍（3 张中间图），逐位置相加，再加偏置：\n")

feat = np.zeros((C_out, 3, 3))
for c in range(C_out):
    maps = []
    for ch in range(C_in):
        m = conv2d_valid(img3[ch], kernels3[c, ch])    # 中间打分图 (3, 3)
        maps.append(m)
        print(f"  输出通道 {c} ← 输入通道 {ch}：中间图形状 {m.shape}")
    feat[c] = np.sum(maps, axis=0) + biases3[c]
    print(f"  输出通道 {c} = 三张中间图逐位置相加 + 偏置 {biases3[c]:+.1f}\n")

print("输出形状   feat    =", feat.shape, "  ← [C_out=2, 3, 3]")
n_formula = (k * k * C_in + 1) * C_out
print(f"\n参数量核对：公式 (K·K·C_in + 1) × C_out = ({k}×{k}×{C_in} + 1) × {C_out} = {n_formula}")
print(f"实际数组元素：kernels {kernels3.size} 个 + biases {biases3.size} 个 = {kernels3.size + biases3.size} ✓")

**小结**：核是 K×K×C_in 的小方块，输出在通道维上"混合"了输入的全部通道——下一层因此能组合"边缘 + 颜色"这类跨通道特征；参数量公式 (K·K·C_in+1)×C_out 与实际数组逐个对上（3×3×3+1)×2 = 56。

## 实验 3：卷积核动物园——同一张图过 6 种核

**目标**：自造一张 64×64 几何图（圆 + 方块 + 45° 斜线），依次过 Sobel 竖/横、锐化、模糊、拉普拉斯、浮雕 6 种经典核，铺成 2×3 特征图墙——每种核"看"到的东西不同。交互版见 [playground 卷积可视化](../playground/conv.html)。注意：真实 CNN 的核不是手工设计的，是随机初始化后训练学出来的（正文误区一）。

In [ ]:
# ========= 实验 3：卷积核动物园 =========
H = W = 64
yy, xx = np.mgrid[0:H, 0:W]                            # yy=行号, xx=列号
img = np.zeros((H, W))
img[((xx - 46) ** 2 + (yy - 18) ** 2) <= 8 ** 2] = 1.0  # 圆（右上）
img[36:58, 6:28] = 1.0                                  # 方块（左下）
img[np.arange(6, 60), np.arange(6, 60)] = 1.0           # 45° 斜线（对角方向）

kernels = {
    "Sobel 竖边缘": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float),
    "Sobel 横边缘": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], float),
    "锐化":         np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], float),
    "模糊（均值）": np.ones((3, 3)) / 9.0,
    "拉普拉斯":     np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], float),
    "浮雕":         np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], float),
}

plt.figure(figsize=(3.8, 3.8))
plt.imshow(img, cmap="gray")
plt.title("自造几何图：圆 + 方块 + 斜线")
plt.xlabel("列 →"); plt.ylabel("行 ↓")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (name, K) in zip(axes.ravel(), kernels.items()):
    fm = conv2d_valid(img, K)
    ax.imshow(fm, cmap="gray")
    ax.set_title(f"{name}（输出 {fm.shape[0]}×{fm.shape[1]}）")
fig.suptitle("卷积核动物园：同一张图，六种『看法』", fontsize=13)
plt.tight_layout(); plt.show()

print("读图指南：")
print("  Sobel 竖边缘只点亮竖直的明暗交界（圆的左右两侧、方块的左右边），横边缘只点亮水平交界；")
print("  锐化把交界两侧的反差拉大（图看起来更『硬』）；模糊把细节抹平；")
print("  拉普拉斯对任何方向的交界都响应（各向同性）；浮雕把明暗交界变成浮雕光影。")

**小结**：一枚核 = 一种"看图方式"：Sobel 两枚分别盯竖/横交界（呼应实验 1 的方向选择性），拉普拉斯各向同性，锐化/模糊/浮雕各有性格。真实 CNN 第一层学出的核长得就像这些边缘/斑点检测器——但它们是从数据里学出来的，不是人设计的。交互版本：[playground/conv.html](../playground/conv.html)。

## 实验 4：最大池化——给"打分图"做缩略图

**目标**：拿实验 3 的 Sobel 竖边缘响应当打分图，做 2×2 最大池化：形状减半、元素个数掉到 1/4，"哪里有边缘"的方位信息大体保留；再叠一层看连续缩略。顺带做 1 像素小平移实验，看池化对平移的"钝感"（小幅平移不变性）。

In [ ]:
# ========= 实验 4：最大池化 =========
def maxpool2(image):
    """2×2 最大池化：每个小块只留最大值，尺寸减半"""
    h, w = image.shape
    return image[: h // 2 * 2, : w // 2 * 2].reshape(h // 2, 2, w // 2, 2).max(axis=(1, 3))

edge_map = np.abs(conv2d_valid(img, kernels["Sobel 竖边缘"]))   # 打分图（取绝对值：正负响应都是『有边缘』）
pool1 = maxpool2(edge_map)
pool2 = maxpool2(pool1)
print(f"打分图 {edge_map.shape} → 池化一次 {pool1.shape} → 池化两次 {pool2.shape}"
      f"（元素个数 {edge_map.size} → {pool1.size} → {pool2.size}，后两层计算量大幅下降）")

# 小幅平移实验：把原图右下挪 1 像素，对比池化前/后输出的相对变化
img_shift = np.roll(np.roll(img, 1, axis=0), 1, axis=1)
edge_shift = np.abs(conv2d_valid(img_shift, kernels["Sobel 竖边缘"]))
rel_raw = np.abs(edge_map - edge_shift).sum() / edge_map.sum()
rel_pool = np.abs(pool1 - maxpool2(edge_shift)).sum() / pool1.sum()
print(f"平移 1 像素后：池化前输出相对变化 {rel_raw:.1%}，池化后 {rel_pool:.1%}"
      "（池化后变化更小 = 对小幅平移更钝感）")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(edge_map, cmap="gray")
axes[0].set_title(f"池化前（{edge_map.shape[0]}×{edge_map.shape[1]}）")
axes[1].imshow(pool1, cmap="gray")
axes[1].set_title(f"2×2 最大池化一次（{pool1.shape[0]}×{pool1.shape[1]}）")
axes[2].imshow(pool2, cmap="gray")
axes[2].set_title(f"池化两次（{pool2.shape[0]}×{pool2.shape[1]}）")
for ax in axes:
    ax.set_xlabel("列 →"); ax.set_ylabel("行 ↓")
fig.suptitle("最大池化：缩略图保方位——亮的地方还在原来的方向", fontsize=13)
plt.tight_layout(); plt.show()

**小结**：池化把 62×62 缩到 31×31 再到 15×15，元素个数掉到约 1/16，但"边缘在哪个方位"仍一眼可读——正文"看缩略图也认得出是猫"的现场。平移实验里池化后的相对变化更小：小幅平移常常整块被同一个 max 吃掉，这就是池化带来的（近似的）平移不变性。注意池化的主作用是降维、扩感受野、带平移不变性，防过拟合主要还得靠增广与正则（正文误区二）。

## 实验 5：感受野计算器——递推公式 r_l = r_{l-1} + (k_l−1)·j_{l-1}

**目标**：把正文"数学深潜"的归纳证明变成可跑的计算器：给一串 (核大小 k, 步幅 s)，逐层打印感受野 r 与跳距 j 的记账过程。先复算正文 SmallCNN 的 3→4→8→10，再算 VGG 式"小核深堆"——三层 3×3 的感受野 7 等于一层 7×7，参数却只有 27 vs 49。

In [ ]:
# ========= 实验 5：感受野计算器 =========
def receptive_field(layers):
    """逐层套正文递推：r ← r + (k-1)·j（多看 k 个位置、每个跨 j 个输入像素）；j ← j·s（跳距累乘）"""
    r, j = 1, 1
    print("  层配置            感受野 r 的记账                跳距 j")
    print("  输入              r = 1                          j = 1")
    for i, (k, s) in enumerate(layers, 1):
        r_new = r + (k - 1) * j
        j_new = j * s
        print(f"  第{i}层 k={k}, s={s}       r = {r} + ({k}-1)×{j} = {r_new:<3d}           j = {j}×{s} = {j_new}")
        r, j = r_new, j_new
    return r, j

print("配置 A：正文 SmallCNN（卷积→池化→卷积→池化）")
rA, jA = receptive_field([(3, 1), (2, 2), (3, 1), (2, 2)])
assert (rA, jA) == (10, 4)
print(f"  → 最终感受野 {rA}×{rA}、跳距 {jA}（与正文表格逐行一致 ✓）\n")

print("配置 B：VGG 式小核深堆（三层 3×3 卷积、步幅 1）")
rB, jB = receptive_field([(3, 1), (3, 1), (3, 1)])
assert rB == 7
print(f"  → 三层 3×3 的感受野 7×7 = 一层 7×7；参数 3×(3×3) = 27 < 一层 7×7 的 49，还多两次非线性\n")

print("配置 C：步幅当加速器（卷积/池化交替六层）")
rC, jC = receptive_field([(3, 1), (2, 2), (3, 1), (2, 2), (3, 1), (2, 2)])
print(f"  → 六层就看到 {rC}×{rC}：池化把跳距翻倍，是感受野的『加速器』（正文推论）")

**小结**：记账过程就是正文闭式解 r = 1 + Σ(k_i−1)·j_{i-1} 的逐项累加；SmallCNN 的 3→4→8→10、VGG 的"小核深堆更省参数"全部复算一致。边界提醒（正文反例）：一旦某层 k < s（核比步幅小），相邻区间出现"缝"，公式算的就成了"跨度"而不是"覆盖"——空洞卷积正是故意挖缝。

## 实验 6：小 CNN 训手写数字——torch 版（需本地）+ sklearn 离线替身

**目标**：正文 SmallCNN（Conv→ReLU→Pool 两组 → Linear，20490 参数）在 MNIST 上训练，并逐层打印形状流。

> **6A 需本地 PyTorch 环境，代码已按标准写法给出**；未安装时自动跳过。**本地运行预期**：参数 20490；2 个 epoch 后 MNIST 测试集准确率 98%+（CPU 数分钟）；真实 MNIST 首次运行需联网下载约 11 MB。6B 是离线替身：sklearn 内置 8×8 digits + MLP，本环境实跑约 98%（预期 95%+）。

In [ ]:
# ========= 实验 6A（需本地 PyTorch 环境）：SmallCNN 训 MNIST =========
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader
    from torchvision import datasets, transforms
    HAS_TORCH = True
except Exception:                # 未安装或安装损坏（如 DLL 加载失败）都优雅跳过
    HAS_TORCH = False
    print("未检测到可用的 PyTorch → 跳过 6A 实跑（本地 pip install torch torchvision 后重跑本单元）")

if HAS_TORCH:
    torch.manual_seed(0)                              # 固定随机种子，结果可复现

    # ---- 数据：真实 MNIST（首跑自动下载约 11MB，需联网） ----
    transform = transforms.Compose([
        transforms.ToTensor(),                       # 像素缩放到 [0, 1]
        transforms.Normalize((0.1307,), (0.3081,)),  # MNIST 的均值/标准差
    ])
    train_set = datasets.MNIST("./data", train=True, download=True, transform=transform)
    test_set = datasets.MNIST("./data", train=False, transform=transform)
    train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=256)

    # ---- 模型：两组「卷积 → ReLU → 最大池化」+ 全连接（与正文 SmallCNN 完全一致） ----
    class SmallCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(1, 16, kernel_size=3, padding=1),    # [16, 28, 28]
                nn.ReLU(),
                nn.MaxPool2d(2),                               # [16, 14, 14]
                nn.Conv2d(16, 32, kernel_size=3, padding=1),   # [32, 14, 14]
                nn.ReLU(),
                nn.MaxPool2d(2),                               # [32, 7, 7]
                nn.Flatten(),                                  # 32×7×7 = 1568
                nn.Linear(32 * 7 * 7, 10),
            )

        def forward(self, xb):
            return self.net(xb)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SmallCNN().to(device)
    print("参数总量：", sum(p.numel() for p in model.parameters()))  # 预期 20490 = 160 + 4640 + 15690

    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    # ---- 训练循环：复用第 4 章的四步模板 ----
    for epoch in range(1, 3):                          # 教学演示跑 2 轮
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)   # ① 前向 + 打分
            optimizer.zero_grad()           # ② 清梯度
            loss.backward()                 # ③ 反向传播
            optimizer.step()                # ④ 更新权重
        print(f"epoch {epoch}  最后一批 loss = {loss.item():.4f}")

    # ---- 评估 ----
    model.eval()
    correct = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            correct += (model(xb.to(device)).argmax(1) == yb.to(device)).sum().item()
    print(f"MNIST 测试集准确率 = {correct / len(test_set):.2%}（本地预期 98%+）")

    # ---- 形状流：喂一张假图逐层打印（与实验 2 的 numpy 版逐层呼应） ----
    with torch.no_grad():
        probe = torch.zeros(1, 1, 28, 28)
        for layer in model.net:
            probe = layer(probe)
            print(f"{layer.__class__.__name__:<10s} → {tuple(probe.shape)}")

In [ ]:
# ========= 实验 6B（本机可跑的离线替身）：sklearn digits + MLP =========
# digits 是 sklearn 内置的 8×8 手写数字集（1797 张、无需联网），相当于迷你版 MNIST
import warnings
from sklearn.datasets import load_digits
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)   # 屏蔽未收敛提示，不影响结果

digits = load_digits()
Xd, yd = digits.data, digits.target                    # (1797, 64) = 8×8 展平
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    Xd, yd, test_size=0.25, random_state=42, stratify=yd)
scaler = StandardScaler().fit(Xd_tr)                   # 标准化：MLP 对尺度敏感

mlp = MLPClassifier(hidden_layer_sizes=(128,), activation="relu", solver="adam",
                    learning_rate_init=1e-3, max_iter=500, random_state=42)
mlp.fit(scaler.transform(Xd_tr), yd_tr)
pred = mlp.predict(scaler.transform(Xd_te))
acc = accuracy_score(yd_te, pred)
print(f"digits 测试集准确率 = {acc:.2%}（预期 95%+；8×8 分辨率低于 MNIST 的 28×28，天花板也低）")

fig, axes = plt.subplots(2, 5, figsize=(10, 4.6))
for ax, i in zip(axes.ravel(), range(10)):
    ax.imshow(Xd_te[i].reshape(8, 8), cmap="gray_r")
    ok = "✓" if pred[i] == yd_te[i] else "✗"
    ax.set_title(f"{ok} 预测 {pred[i]} / 真实 {yd_te[i]}", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("MLP 在 8×8 digits 上的预测样例", fontsize=13)
plt.tight_layout(); plt.show()

**小结**：torch 单元本地预期：参数 20490（Conv1 160 + Conv2 4640 + Linear 15690，大头在全连接层——正文"全局平均池化替代展平"的动机）、2 个 epoch 测试准确率 98%+，形状流 (1,16,28,28)→(1,16,14,14)→(1,32,14,14)→(1,32,7,7)→(1,1568)→(1,10) 与实验 2 的 numpy 形状流逐层对应。离线替身（8×8 digits + MLP）实跑约 98%（预期 95%+），"训练循环 → 收敛 → 看预测样例"的全流程一致。CNN 比 MLP 强在哪？参数更少还自带"局部 + 平移"两大先验——正是本章主题。

## 实验 7：残差连接的梯度高速公路——20 层网络的梯度范数对比

**目标**：正文⑧折叠框的数学 ∂L/∂x = ∂L/∂y·(∂F/∂x + 1) 变成实测：搭两个 20 层 sigmoid 网络，唯一区别是有无 y = F(x) + x 的 "+x"。反向传播逐层打印梯度范数——普通网络梯度指数衰减（sigmoid 导数 ≤ 0.25 的连乘），残差网络靠 "+1" 直通道把梯度保住。

In [ ]:
# ========= 实验 7：残差连接的梯度对比 =========
def sigmoid(z):
    """sigmoid 激活：把任意实数压到 (0, 1)"""
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_grad(z):
    """σ'(z) = σ(z)(1 - σ(z))，峰值仅 0.25 —— 梯度消失的元凶"""
    s = sigmoid(z)
    return s * (1 - s)

rng = np.random.default_rng(0)
d, n_layers = 16, 20
# 同一套权重喂给两张网络（公平对照）；初始化幅度 ≈ 方差守恒
Ws = [rng.normal(0, 1.0, (d, d)) / np.sqrt(d) for _ in range(n_layers)]
x_in = rng.normal(0, 1.0, d)

def forward_backward(residual):
    """前向存激活，再从输出端逐层回传梯度；返回每层『输入处』的梯度范数（按输入→输出排序）"""
    acts = [x_in]                                      # acts[l] = 第 l 层的输入
    zs = []
    for W in Ws:
        z = W @ acts[-1]
        zs.append(z)
        a = sigmoid(z)
        acts.append(acts[-1] + a if residual else a)   # 有无 "+x" 的唯一区别
    g = acts[-1].copy()                                # 损失 L = 0.5·||输出||²，∂L/∂输出 = 输出
    norms = [np.linalg.norm(g)]                        # 从输出端往输入端收集
    for l in range(n_layers - 1, -1, -1):
        local = Ws[l].T @ (g * sigmoid_grad(zs[l]))    # F 分支回传的梯度
        if residual:
            g = g + local                              # "+1" 直通道：上游梯度原样流回
        else:
            g = local
        norms.append(np.linalg.norm(g))
    return norms[::-1]                                 # 反转成输入端 → 输出端的顺序

norms_plain = forward_backward(residual=False)
norms_res = forward_backward(residual=True)

print("层（输入→输出）   普通网络 ‖∂L/∂x‖     残差网络 ‖∂L/∂x‖")
for l in range(n_layers + 1):
    print(f"  第{l:>2} 层输入       {norms_plain[l]:.3e}            {norms_res[l]:.3e}")
ratio = norms_res[0] / norms_plain[0]
print(f"\n最靠近输入的层：普通网络梯度范数 {norms_plain[0]:.2e} vs 残差 {norms_res[0]:.2e}（相差 {ratio:.0f} 倍）")
print("→ 普通网络梯度逐层指数衰减（sigmoid 导数 ≤ 0.25 的连乘，第 4 章公式现场）；")
print("  残差网络每层多一条 '+1' 直通道，梯度范数全程保持 O(1) ——『高速公路』的实测证据")

plt.figure(figsize=(8.5, 4.8))
plt.semilogy(range(n_layers + 1), norms_plain, "o-", label="普通网络（无残差）")
plt.semilogy(range(n_layers + 1), norms_res, "s-", label="残差网络 y = F(x) + x")
from matplotlib.ticker import FuncFormatter
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))   # 纯文本刻度
plt.xlabel("层号（从输入端到输出端）")
plt.ylabel("该层输入处的梯度 L2 范数（对数轴）")
plt.title("20 层 sigmoid 网络：残差连接给梯度修了条不衰减的高速公路")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**小结**：普通 20 层网络靠近输入的梯度掉到 1e-13 量级（sigmoid 导数连乘的指数衰减）；残差版全程保持 O(1)——∂y/∂x 里的 "+1" 给梯度修了条不衰减的直通道，哪怕 F 分支梯度为 0，误差信号也原样流回浅层。这与 LSTM 记忆单元、U-Net 跳跃连接是同一招：给信息与梯度修旁路。

## 改参数建议（拿这个 notebook 当实验台）

1. **换输入试方向性**：实验 1 把输入转置（`I.T`）再过两枚核——竖核变全 0、横核变亮，方向选择性对调；再把核里的 -1/1 改成 -2/2，看响应强度如何翻倍。
2. **给卷积加步幅/补零**：给实验 2 的 `conv2d_valid` 加上 padding 与 stride 参数，对照输出尺寸公式 ⌊(I−K+2P)/S⌋+1 逐一验证；复现正文"失败现场"：5×5 输入、3×3 核、S=3 时 (5−3)/3+1 取整丢窗口。
3. **动物园添新动物**：实验 3 自己设计核——只盯 45° 斜线的旋转版 Sobel（把竖边缘核旋转 45°）、高斯模糊 [[1,2,1],[2,4,2],[1,2,1]]/16，看它们在圆/方块/斜线上的响应差别；再把图换成 `np.random.rand(64,64)` 的噪声图，看哪些核还能"看出东西"。
4. **感受野玩极端**：实验 5 往配置里加一个 (k=2, s=4) 的层（k < s），对照闭式解想想哪里开始出现"缝"（蜂窝状覆盖，空洞卷积的原理）；或把三层 3×3 换成一层 7×7，参数从 27 涨到 49、非线性少两次。
5. **本地换结构再训**：实验 6A 本地把 `kernel_size` 3→5（padding 同步改 2），参数量 20490→28938；把第一个卷积 padding=1 改 0，Flatten 维度对不上会直接报错（正文"改什么参数"清单的现场）；6B 里把 `hidden_layer_sizes` 换成 (256, 128) 两层，看准确率与训练时间的取舍。